In [1]:
# from google.colab import drive
# drive.mount('/content/drive')
import os
os.environ["QT_QPA_PLATFORM"] = "offscreen"
os.listdir("/home/sraoham1/Illicit")

['Baseline-GNNs.ipynb',
 'Baseline-ML.ipynb',
 '.ipynb_checkpoints',
 'shap_decision_plot_node28207.pdf',
 'plot',
 'revise',
 'AblationStudy.ipynb',
 'top10_feature_importance_node23607.pdf',
 'Ellptic_TAGCN.ipynb',
 'data',
 'result']

In [2]:
#import os
# os.chdir('/content/drive/MyDrive/Graph_Neural_Network')

In [3]:
!pip install torch_geometric==2.5.0
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
from torch_geometric.data import Data
from torch_geometric.nn import TAGConv
from sklearn.model_selection import train_test_split

Defaulting to user installation because normal site-packages is not writeable


In [4]:
df_classes = pd.read_csv('./data/elliptic_bitcoin_dataset/elliptic_txs_classes.csv')
df_edges = pd.read_csv('./data/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv')
df_features = pd.read_csv('./data/elliptic_bitcoin_dataset/elliptic_txs_features.csv', header=None)

In [5]:
# Remove the 'unknown' class, keeping only licit/illicit transactions (1/0)
df_classes = df_classes[df_classes['class'] != 'unknown']
df_classes['class'] = df_classes['class'].map({'1': 1, '2': 0})  # 1: illicit, 0: licit

In [6]:
# Merge features with class labels
df_merge = df_features.merge(df_classes, how='inner', right_on="txId", left_on=0)
df_merge = df_merge.drop(['txId'], axis=1)

In [7]:
# ====================== Graph Construction ======================

# Node IDs
nodes = df_merge[0].values
map_id = {j: i for i, j in enumerate(nodes)}

# Filter edges to include only nodes present in the dataset
edges = df_edges[df_edges.txId1.isin(map_id) & df_edges.txId2.isin(map_id)].copy()
edges.txId1 = edges.txId1.map(map_id)
edges.txId2 = edges.txId2.map(map_id)
edges = edges.astype(int)

# Create edge index for PyTorch Geometric (shape: [2, num_edges])
edge_index = np.array(edges.values).T
edge_index = torch.tensor(edge_index, dtype=torch.long).contiguous()

# Assign uniform weights to all edges
weights = torch.tensor([1] * edge_index.shape[1], dtype=torch.float32)

# Node labels
labels = torch.tensor(df_merge['class'].values, dtype=torch.float32)

# Node features
node_features = torch.tensor(np.array(df_merge.drop([0, 'class', 1], axis=1).values), dtype=torch.float32)

# Create PyG Data object for the graph
elliptic_dataset = Data(x=node_features, edge_index=edge_index, edge_weights=weights, y=labels)

In [8]:
# ====================== Hyperparameters and Dataset Split ======================
seed = 0

# All nodes have labels; prepare indices for train/validation/test sets
all_idx = np.arange(len(labels))
y_all = labels[all_idx]

# Split the dataset into train/validation/test sets with a 70:15:15 ratio
train_idx, temp_idx = train_test_split(
    all_idx, test_size=0.3, random_state=seed, stratify=y_all
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=seed, stratify=labels[temp_idx]
)

# Assign splits to the PyG Data object
elliptic_dataset.train_idx = torch.tensor(train_idx, dtype=torch.long)
elliptic_dataset.val_idx = torch.tensor(val_idx, dtype=torch.long)
elliptic_dataset.test_idx = torch.tensor(test_idx, dtype=torch.long)

In [9]:
# ----------------- Model Definition -----------------
class TGNN(torch.nn.Module):
    def __init__(self, dim_in, dim_h, dim_out, K=3):
        super().__init__()
        self.norm1 = torch.nn.BatchNorm1d(dim_in)
        self.gat1 = TAGConv(dim_in, dim_h, K)
        self.norm2 = torch.nn.BatchNorm1d(dim_h)
        self.gat2 = TAGConv(dim_h, dim_out, K)

    def forward(self, x, edge_index):
        h = self.norm1(x)
        h = self.gat1(h, edge_index)
        h = self.norm2(h)
        h = F.leaky_relu(h)
        out = self.gat2(h, edge_index)
        return out

def train(model, data, criterion, optimizer, num_epochs, checkpoint_dir, model_filename):
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    best_loss = float('inf')
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        pred = model(data.x, data.edge_index)
        loss = criterion(pred[data.train_idx], data.y[data.train_idx].unsqueeze(1))
        loss.backward()
        optimizer.step()

        val_loss = criterion(pred[data.val_idx], data.y[data.val_idx].unsqueeze(1))
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save({'state_dict': model.state_dict()}, os.path.join(checkpoint_dir, model_filename))
     # Load the best model weights
    checkpoint = torch.load(os.path.join(checkpoint_dir, model_filename))
    model.load_state_dict(checkpoint['state_dict'])
    return model

def test(model, data):
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index).squeeze(1)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).long().cpu().numpy()
        labels = data.y.cpu().numpy()
    return preds, labels

def evaluate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    return acc, prec, rec, f1, mcc

In [10]:
# ----------------- Ablation Experiment Loop -----------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_dim = 165
hidden_size = 150
output_dim = 1
learning_rate = 0.005
weight_decay = 1e-5
num_epochs = 400
checkpoints_dir = './result/models/elliptic_tgnn_ablation'

# Move the data to the device (CPU or GPU)
data_train = elliptic_dataset.to(device)


results = []
for K_hop in range(1, 6): 
    print(f'\n==== Training TAGCN with K={K_hop} ====')
    model = TGNN(input_dim, hidden_size, output_dim, K=K_hop).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()

    model = train(model, data_train, criterion, optimizer, num_epochs,
                  checkpoints_dir, f'tgnn_best_model_K{K_hop}.pth.tar')
    #model.load_state_dict(torch.load(os.path.join(checkpoints_dir, 'tgnn_best_model_K%s.pth.tar' % K_hop))['state_dict'])


    preds, labels = test(model, data_train)
    # Select only the nodes in the test set
    test_mask = elliptic_dataset.test_idx.cpu().numpy()
    y_true = labels[test_mask]
    y_pred = preds[test_mask]

    acc, prec, rec, f1, mcc = evaluate_metrics(y_true, y_pred)
    print(f'K={K_hop} Test Metrics: Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f} MCC={mcc:.4f}')
    results.append({
        'K-hop': K_hop,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-score': f1,
        'MCC': mcc
    })

# Convert to DataFrame for display
df_results = pd.DataFrame(results)
print('\n===== Ablation Study Results =====')
print(df_results.to_string(index=False))


==== Training TAGCN with K=1 ====
K=1 Test Metrics: Acc=0.9810 Prec=0.9435 Rec=0.8563 F1=0.8978 MCC=0.8886

==== Training TAGCN with K=2 ====
K=2 Test Metrics: Acc=0.9797 Prec=0.9259 Rec=0.8607 F1=0.8921 MCC=0.8816

==== Training TAGCN with K=3 ====
K=3 Test Metrics: Acc=0.9804 Prec=0.9346 Rec=0.8592 F1=0.8953 MCC=0.8855

==== Training TAGCN with K=4 ====
K=4 Test Metrics: Acc=0.9785 Prec=0.9277 Rec=0.8460 F1=0.8850 MCC=0.8743

==== Training TAGCN with K=5 ====
K=5 Test Metrics: Acc=0.9797 Prec=0.9327 Rec=0.8534 F1=0.8913 MCC=0.8811

===== Ablation Study Results =====
 K-hop  Accuracy  Precision   Recall  F1-score      MCC
     1  0.980959   0.943457 0.856305  0.897771 0.888567
     2  0.979671   0.925868 0.860704  0.892097 0.881607
     3  0.980387   0.934609 0.859238  0.895340 0.885492
     4  0.978525   0.927653 0.846041  0.884969 0.874280
     5  0.979671   0.932692 0.853372  0.891271 0.881140


In [11]:
import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd

# Keep only the Recall, F1-score, and MCC metrics
df_long = df_results.melt(
    id_vars=['K-hop'],
    value_vars=['Recall', 'F1-score', 'MCC'],
    var_name='Metric',
    value_name='Score'
)

# Plot a line chart
fig = px.line(
    df_long,
    x='K-hop',
    y='Score',
    color='Metric',
    markers=True,
    line_shape='linear'
)

# Control figure size (width and height) and layout
# fig.update_layout(
#     width=1000,     # Set figure width, e.g., 1000 pixels
#     height=500,     # Set figure height
#     xaxis=dict(
#         title='K-hop',
#         dtick=1,
#         titlefont=dict(size=16),
#         tickfont=dict(size=14)
#     ),
#     yaxis=dict(
#         title='Metric Value',
#         range=[0.84, 0.91],
#         titlefont=dict(size=16),
#         tickfont=dict(size=14)
#     ),
#     legend_title_text='Evaluation Metric',
#     legend=dict(font=dict(size=14)),
#     template='plotly_white'
# )
# fig.show()
fig.update_layout(
    width=1000,
    height=500,
    xaxis=dict(
        title=dict(text='K-hop', font=dict(size=16)),
        dtick=1,
        tickfont=dict(size=14)
    ),
    yaxis=dict(
        title=dict(text='Metric Value', font=dict(size=16)),
        range=[0.84, 0.91],
        tickfont=dict(size=14)
    ),
    legend_title_text='Evaluation Metric',
    legend=dict(font=dict(size=14)),
    template='plotly_white'
)